# Clase 5 · Variantes opcionales

**Construimos:** dos o tres variantes del principal. **Reutilizamos:** su código, proporcionado aquí. **Resultado:** comprobar qué cambia al variar un dato. Todo este notebook es OPTIONAL.

**Recupera:** ¿Qué devuelve la pieza del principal y qué datos conserva?

Guarda una copia `mi_auxiliar.ipynb`. Completa y ejecuta con **Shift+Enter**. Las bases plegadas se ejecutan; las soluciones desplegables son solo texto. Al terminar, reinicia el kernel y ejecuta todo sobre tu copia completada.

<a id="L05-A15"></a>

### Variante 1 · Otro mark

Compra dos unidades a 101 en la cartera preparada. Guarda su equity a 98 y predice qué permanece igual.

**Base proporcionada.** OrderBook, Level, Fill y PositionTracker canónicos del principal; la primera variante crea la cartera y la venta parcial continúa su estado.

In [ ]:
class Level:
    def __init__(self, price, size):
        self.price = price
        self.size = size
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda level: -level.price)
        self.asks = sorted(asks, key=lambda level: level.price)

    @property
    def mid(self):
        if not self.bids or not self.asks:
            return None
        return (self.bids[0].price + self.asks[0].price) / 2

    def imbalance(self, levels=1):
        bid = sum(level.size for level in self.bids[:levels])
        ask = sum(level.size for level in self.asks[:levels])
        return (bid - ask) / (bid + ask) if bid + ask else None

from math import isfinite

fill_record = {'side': 'buy', 'price': 100, 'size': 2}
class Fill:
    def __init__(self, side, price, size):
        valid_numbers = all(isfinite(x) and x > 0 for x in (price, size))
        if side not in ('buy', 'sell') or not valid_numbers:
            raise ValueError('fill requiere lado válido y valores positivos finitos')
        self.side = side
        self.price = price
        self.size = size

    def cash_flow(self):
        sign = -1 if self.side == 'buy' else 1
        return sign * self.price * self.size


class PositionTracker:
    def __init__(self, cash=0.0):
        self.cash = cash
        self.position = 0.0

    def apply_fill(self, fill):
        self.cash += fill.cash_flow()
        self.position += fill.size if fill.side == 'buy' else -fill.size

    def equity(self, mark):
        return self.cash + self.position * mark

tracker = PositionTracker(1000)

In [ ]:
tracker.apply_fill(Fill('buy', 101, 2))
value = None

In [ ]:
print(tracker.cash, tracker.position, value)

<details>
<summary>Pista</summary>

Usa equity(98) sobre la cartera comprada.

</details>

<details>
<summary>Solución</summary>

```python
tracker.apply_fill(Fill('buy', 101, 2))
value = tracker.equity(98)
```

La bajada del mark cambia la valoración, no las unidades.

**Observa:** `798 2.0 994.0`.

</details>

<a id="L05-A16"></a>

### Variante 2 · Venta parcial

Conserva la cartera anterior; vende 0.5 unidades a 103. Predice caja y posición; valora a 100.

In [ ]:
# Aplica el fill de venta parcial.

In [ ]:
print(tracker.cash, tracker.position, tracker.equity(100))

<details>
<summary>Pista</summary>

El tamaño confirmado es 0.5, aunque tu posición sea 2.

</details>

<details>
<summary>Solución</summary>

```python
tracker.apply_fill(Fill('sell', 103, .5))
```

La cartera contabiliza el fill parcial; no cierra automáticamente la posición.

**Observa:** `849.5 1.5 999.5`.

</details>

<a id="L05-B03"></a>

### Ampliación · Profundidad de la medida

Consulta imbalance con uno y dos niveles. Anticipa si cambiará su signo.

**Base proporcionada.** Datos del ejemplo; ejecuta esta celda antes de completar los huecos.

In [ ]:
book = OrderBook([Level(99,1), Level(100,3)], [Level(102,7), Level(101,1)])

In [ ]:
one_level = None
two_levels = None

In [ ]:
print(one_level, two_levels)

<details>
<summary>Pista</summary>

Usa imbalance(1) e imbalance(2).

</details>

<details>
<summary>Solución</summary>

```python
one_level = book.imbalance(1)
two_levels = book.imbalance(2)
```

Los mismos Level; cambia cuánta profundidad consultas.

**Observa:** `0.5` y aproximadamente `-0.3333333333`.

</details>

<a id="L05-A14"></a>

### Ampliación · Propiedad actual

Guarda `before=book.mid`, cambia el primer ask a 106 y guarda `after`. ¿Se actualiza una variable que guardaste antes?

**Base proporcionada.** Datos del ejemplo; ejecuta esta celda antes de completar los huecos.

In [ ]:
book = OrderBook([Level(100,1)], [Level(102,1)])

In [ ]:
before = None
# Cambia book.asks[0].price.
after = None

In [ ]:
print(before, after, book.mid)

<details>
<summary>Pista</summary>

Modifica el atributo price del Level, no su representación.

</details>

<details>
<summary>Solución</summary>

```python
before = book.mid
book.asks[0].price = 106
after = book.mid
```

La variable before conserva el número antiguo; la propiedad vuelve a calcular.

**Observa:** `101.0 103.0 103.0`.

</details>

<a id="L05-A13"></a>

### Ampliación · Tamaño comprador

Suma los tamaños de los bids con un bucle. No sumes precios ni asks.

**Base proporcionada.** Datos del ejemplo; ejecuta esta celda antes de completar los huecos.

In [ ]:
book = OrderBook([Level(100,1),Level(99,2),Level(98,3)], [Level(101,8)])

In [ ]:
depth = 0

In [ ]:
print(depth)

<details>
<summary>Pista</summary>

Acumula level.size.

</details>

<details>
<summary>Solución</summary>

```python
depth = 0
for level in book.bids:
    depth += level.size
```

El tamaño total está en unidades, no en dinero.

**Observa:** `6`.

</details>

Guarda el notebook. Explica qué cambiaste y por qué produce esos resultados.